In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('/media/carsen/ssd1/github/oneshot')
from utils import data

In [ ]:
# FECV
# load text16 data
mouse_ids = [7,8,10,11,12]
mouse_id = mouse_ids[2]
save_path = './result'
fname = 'text16_%s_%s.npz'%(data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
dat = np.load(os.path.join('../data', fname), allow_pickle=True)

txt16_spks_train = dat['sp'].T
# txt16_istim_train = dat['istim'].astype(int)
txt16_labels_train = dat['labels']

print('txt16_spks_train shape:', txt16_spks_train.shape)
print('txt16_labels_train shape:', txt16_labels_train.shape)


txt16_spks_test_ori = dat['ss_all'].astype('float') # each image has different reps
print('txt16_spks_test shape (nstim, nrep, nneuron):', txt16_spks_test_ori.shape)
# txt16_istim_test_ori = dat['ss_istim'].astype(int)
txt16_labels_test_ori = dat['ss_labels']

txt16_spks_test = []
txt16_labels_test = []

for i in range(txt16_spks_test_ori.shape[0]):
    nrep = txt16_spks_test_ori[i].shape[0]
    txt16_spks_test.append(txt16_spks_test_ori[i])
    txt16_labels_test.append(np.repeat(txt16_labels_test_ori[i], nrep))

txt16_spks_test = np.vstack(txt16_spks_test)
txt16_labels_test = np.hstack(txt16_labels_test)

print('txt16_spks_test shape:', txt16_spks_test.shape)
print('txt16_labels_test shape:', txt16_labels_test.shape)

txt16_spks = np.vstack((txt16_spks_train, txt16_spks_test))
txt16_labels = np.hstack((txt16_labels_train, txt16_labels_test)).astype(int)
# txt16_istim = np.hstack((txt16_istim_train, txt16_istim_test))

print('txt16_spks shape:', txt16_spks.shape)
print('txt16_labels shape:', txt16_labels.shape)

from scipy.stats import zscore
txt16_spks = zscore(txt16_spks, axis=0)

from utils import metrics
neural_catvar_all = metrics.category_variance_pairwise(txt16_spks.T, txt16_labels)
print('neural_catvar_all shape:', neural_catvar_all.shape)

In [ ]:
plt.hist(neural_catvar_all, bins=50)

In [ ]:
save_name = f'{data.mouse_names[mouse_id]}_{data.exp_date[mouse_id]}_neural_FECV.npy'
np.save(os.path.join(save_path, save_name), neural_catvar_all)